## M1 Live Loop Driver
Runs a true online M1 loop: live TransformerLens segment generation between `RealCoreEngine` observations.

In [2]:
%pip uninstall -y numpy
%pip install -q numpy==1.26.4
%pip install -q transformer-lens torch matplotlib pandas datasets

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is 

In [14]:
import json
import random
import re
import sys
from datetime import datetime, timezone
from pathlib import Path



exec(open("/content/Phase 5/colab_setup.py").read())


def sanitize_tag(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")

PROMPT_PATH = PHASE5_ROOT / "experiments" / "m0" / "prompts_m0.json"
RESULTS_ROOT = PHASE5_ROOT / "experiments" / "m1" / "live"

RUN_MODE = "full"  # smoke | full
MODEL_NAME_MAP = {
    "qwen3_0_6b": "Qwen/Qwen3-0.6B",
    "tinyllama_1_1b": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "pythia_1b": "EleutherAI/pythia-1b",
    "pythia_2_8b": "EleutherAI/pythia-2.8b",
}
MODEL_KEY = "pythia_1b"
FALLBACK_MODEL_KEYS = ["qwen3_0_6b"]

PROMPT_ID = "cp_001"
SEED = 42
INITIAL_TEMPERATURE = 0.8
SEGMENT_TOKENS = 16 if RUN_MODE == "smoke" else 24
MAX_TOTAL_NEW_TOKENS = 96 if RUN_MODE == "smoke" else 240
TARGET_CYCLES = 8 if RUN_MODE == "smoke" else 20

EXPERIMENT_TAG = f"{sanitize_tag(MODEL_KEY)}_{sanitize_tag(RUN_MODE)}"
RESULTS_DIR = RESULTS_ROOT / EXPERIMENT_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

print("Phase 5 root:", PHASE5_ROOT)
print("Phase 4 root:", PHASE4_ROOT)
print("Results dir:", RESULTS_DIR)
print("Model key:", MODEL_KEY)
print("Prompt id:", PROMPT_ID)

Phase 5 root : /content/Phase 5
Phase 4 root : /content/Phase 4
Phase 5 root: /content/Phase 5
Phase 4 root: /content/Phase 4
Results dir: /content/Phase 5/experiments/m1/live/pythia_1b_full
Model key: pythia_1b
Prompt id: cp_001


In [15]:
import torch
from transformer_lens import HookedTransformer

from real_core.engine import RealCoreEngine
from real_inference import (
    InferenceActionBackend,
    InferenceCoherenceModel,
    InferenceRuntimeState,
    LiveLoopConfig,
    LiveSegmentObservationAdapter,
)


def load_prompt_by_id(prompt_path: Path, prompt_id: str) -> dict:
    data = json.loads(prompt_path.read_text(encoding="utf-8"))
    for p in data["prompts"]:
        if p["id"] == prompt_id:
            return p
    raise ValueError(f"Prompt id not found: {prompt_id}")


def load_model_with_fallback(primary_key: str, fallback_keys: list[str]):
    requested = [primary_key] + [k for k in fallback_keys if k != primary_key]
    errors = []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    for key in requested:
        name = MODEL_NAME_MAP[key]
        try:
            print(f"Loading model [{key}] {name} on {device} ...")
            model = HookedTransformer.from_pretrained(name, device=device, dtype=dtype)
            print(f"Loaded model [{key}] {name}")
            return model, key, name, device, str(dtype)
        except Exception as exc:
            errors.append({"model_key": key, "model_name": name, "error": str(exc)})
            print(f"Failed [{key}]: {exc}")

    raise RuntimeError(f"Unable to load model. Errors: {errors}")


prompt_obj = load_prompt_by_id(PROMPT_PATH, PROMPT_ID)
model, resolved_model_key, resolved_model_name, device_name, dtype_name = load_model_with_fallback(
    MODEL_KEY,
    FALLBACK_MODEL_KEYS,
)
print("Prompt topic:", prompt_obj["topic"])

Loading model [pythia_1b] EleutherAI/pythia-1b on cuda ...


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.09G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model EleutherAI/pythia-1b into HookedTransformer
Loaded model [pythia_1b] EleutherAI/pythia-1b
Prompt topic: remote_work_policy


In [16]:
state = InferenceRuntimeState(current_temperature=INITIAL_TEMPERATURE)
live_cfg = LiveLoopConfig(
    segment_tokens=SEGMENT_TOKENS,
    max_total_new_tokens=MAX_TOTAL_NEW_TOKENS,
    temperature=INITIAL_TEMPERATURE,
    sample_multinomial=True,
)
observer = LiveSegmentObservationAdapter(
    model=model,
    runtime_state=state,
    prompt=prompt_obj["prompt"],
    config=live_cfg,
)
actions = InferenceActionBackend(state)
coherence = InferenceCoherenceModel()

engine = RealCoreEngine(
    observer=observer,
    actions=actions,
    coherence=coherence,
    domain_name="phase5_inference_live_m1",
)

cycles = min(TARGET_CYCLES, observer.max_cycles_estimate)
summary = engine.run_session(cycles=cycles, consolidate_on_action="rest")

print("Session summary:")
print(" cycles:", summary.cycles)
print(" mean_coherence:", round(summary.mean_coherence, 4))
print(" final_coherence:", round(summary.final_coherence, 4))
print(" gco_counts:", summary.gco_counts)
print(" generation_complete:", observer.generation_complete)

Session summary:
 cycles: 10
 mean_coherence: 0.4547
 final_coherence: 0.5491
 gco_counts: {'STABLE': 0, 'PARTIAL': 0, 'DEGRADED': 8, 'CRITICAL': 2}
 generation_complete: False


In [17]:
print("First 5 cycle entries:")
for entry in engine.memory.entries[:5]:
    print(
        {
            "cycle": entry.cycle,
            "action": entry.action,
            "mode": entry.mode,
            "coherence": round(entry.coherence, 4),
            "gco": entry.gco.value,
            "delta": round(entry.delta, 4),
            "entropy_mean": round(float(entry.state_after.get("token_entropy_mean", 0.0)), 4),
            "temp": round(float(entry.state_after.get("temperature", 0.0)), 3),
        }
    )

print("Generated text preview:")
print(observer.generated_text()[:600])

First 5 cycle entries:
{'cycle': 1, 'action': 'rest', 'mode': 'fluctuation', 'coherence': 0.3828, 'gco': 'CRITICAL', 'delta': 0.0, 'entropy_mean': 2.2326, 'temp': 0.8}
{'cycle': 2, 'action': 'observe', 'mode': 'fluctuation', 'coherence': 0.506, 'gco': 'DEGRADED', 'delta': 0.1232, 'entropy_mean': 2.1431, 'temp': 0.8}
{'cycle': 3, 'action': 'observe', 'mode': 'fluctuation', 'coherence': 0.4626, 'gco': 'DEGRADED', 'delta': -0.0433, 'entropy_mean': 2.1898, 'temp': 0.8}
{'cycle': 4, 'action': 'rest', 'mode': 'fluctuation', 'coherence': 0.3677, 'gco': 'CRITICAL', 'delta': -0.0949, 'entropy_mean': 2.5412, 'temp': 0.8}
{'cycle': 5, 'action': 'observe', 'mode': 'constraint', 'coherence': 0.4034, 'gco': 'DEGRADED', 'delta': 0.0357, 'entropy_mean': 2.2982, 'temp': 0.8}
Generated text preview:


You can rely on me to provide you with relevant news and information as well as help you with this project.

As of April 1, 2017, every employee in the US is required to work remotely from some point in th

In [18]:
run_meta = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase5_root": str(PHASE5_ROOT),
    "results_dir": str(RESULTS_DIR),
    "experiment_tag": EXPERIMENT_TAG,
    "run_mode": RUN_MODE,
    "prompt_id": PROMPT_ID,
    "prompt_topic": prompt_obj["topic"],
    "prompt_text": prompt_obj["prompt"],
    "requested_model_key": MODEL_KEY,
    "fallback_model_keys": FALLBACK_MODEL_KEYS,
    "resolved_model_key": resolved_model_key,
    "resolved_model_name": resolved_model_name,
    "device": device_name,
    "dtype": dtype_name,
    "seed": SEED,
    "initial_temperature": INITIAL_TEMPERATURE,
    "segment_tokens": SEGMENT_TOKENS,
    "max_total_new_tokens": MAX_TOTAL_NEW_TOKENS,
    "cycles": cycles,
}

summary_artifact = {
    "cycles": summary.cycles,
    "mean_coherence": summary.mean_coherence,
    "final_coherence": summary.final_coherence,
    "gco_counts": summary.gco_counts,
    "generation_complete": observer.generation_complete,
}

cycle_log = [
    {
        "cycle": e.cycle,
        "action": e.action,
        "mode": e.mode,
        "coherence": e.coherence,
        "delta": e.delta,
        "gco": e.gco.value,
        "state_after": e.state_after,
    }
    for e in engine.memory.entries
]

meta_path = RESULTS_DIR / "run_meta.json"
summary_path = RESULTS_DIR / "m1_live_summary.json"
cycle_path = RESULTS_DIR / "m1_live_cycle_log.jsonl"
segment_path = RESULTS_DIR / "m1_live_segment_trace.jsonl"
text_path = RESULTS_DIR / "generated_text.txt"

meta_path.write_text(json.dumps(run_meta, indent=2), encoding="utf-8")
summary_path.write_text(json.dumps(summary_artifact, indent=2), encoding="utf-8")
text_path.write_text(observer.generated_text(), encoding="utf-8")

with cycle_path.open("w", encoding="utf-8") as f:
    for row in cycle_log:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with segment_path.open("w", encoding="utf-8") as f:
    for row in observer.segment_trace:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Wrote:", meta_path)
print("Wrote:", summary_path)
print("Wrote:", cycle_path)
print("Wrote:", segment_path)
print("Wrote:", text_path)

Wrote: /content/Phase 5/experiments/m1/live/pythia_1b_full/run_meta.json
Wrote: /content/Phase 5/experiments/m1/live/pythia_1b_full/m1_live_summary.json
Wrote: /content/Phase 5/experiments/m1/live/pythia_1b_full/m1_live_cycle_log.jsonl
Wrote: /content/Phase 5/experiments/m1/live/pythia_1b_full/m1_live_segment_trace.jsonl
Wrote: /content/Phase 5/experiments/m1/live/pythia_1b_full/generated_text.txt
